# 03 · Entrenar Tiny Voca en español desde cero (voz F2)

Tercer y último paso. Entrena el modelo con `python -m tiny_voca.train` **del repo oficial, sin cambiar su
código de entrenamiento**, sobre las anotaciones del notebook 02 (`02_alineacion_v2/`). Lo único que se
cambia es `tiny_voca/text.py` (el front end inglés) por un adaptador al front end español
`frontend_es.py` que escribió el 02: mismo inventario de 56 símbolos y misma huella con la que se anotó.

**Necesita GPU** (*Entorno de ejecución → Cambiar tipo de entorno*):

| GPU | Velocidad aproximada | Comentario |
| --- | --- | --- |
| **A100** | la más rápida (~8–10 h para 100 000 pasos) | gasta más unidades por hora |
| **L4** | ~2× más lenta que la A100 | buen equilibrio |
| T4 | ~4× más lenta | barata pero muy lenta |

La celda 3 mide la velocidad real a los pocos minutos y calcula cuánto falta.

**Reanudable:** todo se guarda en Drive (`03_entrenamiento/<RUN_NAME>/`). Si Colab se corta o paras la celda 3,
vuelve a *Ejecutar todo* y sigue en el paso donde iba. **Puedes escuchar el modelo cuando quieras**: para la
celda 3 (botón ■; guarda antes de salir) y ejecuta la 4. Para continuar, ejecuta otra vez la 3.

**Qué esperar:** hasta el paso ~8 000 suena a robot/ruido (aprende fonemas y duraciones). Desde ahí entran
los discriminadores y la voz se vuelve natural poco a poco. A partir de ~40 000 pasos ya debería sonar bien.

In [ ]:
#@title 1 · Configuración
ALIGN_DIR = "/content/drive/MyDrive/tiny-voca-es-desde-cero/02_alineacion_v2"   # salida del notebook 02
RUNS_DIR = "/content/drive/MyDrive/tiny-voca-es-desde-cero/03_entrenamiento"
RUN_NAME = "f2-v2"          # cambia el nombre para empezar otro entrenamiento desde cero (el anterior se queda)

STEPS = 100000              # pasos totales; se puede subir después y seguir (el repo oficial usó 104 000)
SYNC_MINUTES = 10           # copia a Drive cada tanto (además de al terminar o parar)

# Receta del repo oficial (README). Si cambias algo de esto, el entrenamiento ya empezado no se puede
# reanudar: usa otro RUN_NAME.
RECIPE = {"--gan-start": 8000, "--gan-ratio": 0.7, "--teacher-final": 0, "--batch-frames": 9000,
          "--seed": 20261113}
SAVE_EVERY = 1000           # guarda inference.pt (el modelo para escuchar) cada estos pasos
KEEP_EVERY = 20000          # además conserva una copia completa numerada cada estos pasos
RUIDO_FRICATIVAS = True     # celda 4: «s», «f» y respiraciones con ruido en vez de pulsos (menos estática; solo inferencia)

LISTEN_SENTENCES = [        # frases de la celda 4 (números con letra)
    "Hola, soy una voz sintética entrenada desde cero en español.",
    "¿Te gustaría dar un paseo por el parque esta tarde?",
    "La ciudad de México amaneció con lluvia y mucho tráfico.",
    "¡Qué alegría verte de nuevo después de tanto tiempo!",
]

In [ ]:
#@title 2 · Drive, GPU, repo oficial con el front end español y datos en disco local
import json, os, shutil, subprocess, sys, time
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
import torch
if not torch.cuda.is_available():
    raise RuntimeError("No hay GPU: Entorno de ejecución → Cambiar tipo de entorno → GPU (A100, L4 o T4)")
print("GPU:", torch.cuda.get_device_name(0), "·", round(torch.cuda.get_device_properties(0).total_memory / 2**30), "GB")

ALIGN = Path(ALIGN_DIR)
ANNOTATIONS = ALIGN / "anotaciones"
metadata = json.loads((ANNOTATIONS / "annotations.json").read_text())
if not metadata.get("complete"):
    raise RuntimeError("Las anotaciones del 02 no están completas: termina antes el notebook 02 (celda 8)")
RUN = Path(RUNS_DIR) / RUN_NAME
RUN.mkdir(parents=True, exist_ok=True)
RUN_LOCAL = Path("/content/run")
DATA_LOCAL = Path("/content/datos")

# Repo oficial en el mismo commit con el que el 02 preparó los rasgos acústicos.
COMMIT = metadata["identity"]["tiny_voca_commit"]
REPO = Path("/content/tiny-voca")
if not REPO.exists():
    subprocess.run(["git", "clone", "-q", "https://github.com/n4ze3m/tiny-voca.git", str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "-q", COMMIT], check=True)
REPO_SRC = next(p.parent.parent for p in REPO.rglob("tiny_voca/train.py"))
subprocess.run(["apt-get", "-qq", "install", "-y", "espeak-ng"], check=True, stdout=subprocess.DEVNULL)

# Front end español: frontend_es.py tal cual lo dejó el 02 en Drive + adaptador en lugar de tiny_voca/text.py.
TEXT_ES = r'''"""tiny_voca.text para Tiny Voca en español (lo instala el notebook 03 en lugar del original).

Mismos nombres que el front end inglés del repo oficial, pero todo sale de ``frontend_es.py``
(eSpeak NG es-419, 56 símbolos): el entrenamiento, la inferencia y la exportación del repo oficial
funcionan sin cambios y comprueban la misma huella con la que el notebook 02 anotó.
"""
import frontend_es as _fe
from frontend_es import (BOS, CONSONANTS, EOS, PAD, PHONE_IDS, PHONE_SET, PHONES, PUNCTUATION, SEPARATOR, SP,
                         SPECIALS, STRESS_MARKS, SYMBOL_TO_ID, SYMBOLS, UNK, VOICELESS, VOWEL_SET, VOWELS,
                         Phonemized, chunks, espeak_executable, espeak_version, espeak_words, normalize_text,
                         parse_espeak, tokenize)

FRONTEND_VERSION = _fe.VERSION
ESPEAK_VOICE = _fe.VOICE
FRONTENDS = ("espeak",)
FRONTEND_VARIABLE = "TINY_VOCA_FRONTEND"
FALLBACK = {}


class NothingToPronounce(ValueError):
    """El texto no tiene nada que decir: solo signos, emojis y similares."""


def frontend():
    return "espeak"


def set_frontend(name):
    if name not in FRONTENDS:
        raise ValueError(f"este modelo solo tiene el front end español ({FRONTENDS}), no {name!r}")


def frontend_identity():
    return _fe.identity()


def frontend_fingerprint():
    return _fe.fingerprint()


def frontend_runtime():
    return {"espeak_version": espeak_version()}


def canonical(phone):
    phone = _fe.EQUIVALENT.get(phone, phone)
    return (phone,) if phone in PHONE_SET else None


def phonemize(text):
    try:
        return _fe.phonemize(text)
    except ValueError as error:
        if "nada que pronunciar" in str(error):
            raise NothingToPronounce(str(error)) from error
        raise


def is_phone(token_ids):
    return [int(i) in PHONE_IDS for i in token_ids]
'''
RUIDO_FRICATIVAS_PY = r'''"""Ajuste de inferencia para Tiny Voca: «s», «f», respiraciones y pausas con ruido en vez de pulsos.

El generador de onda de tiny-voca excita su rama armónica con un tren de pulsos a la frecuencia de la voz.
Todo lo que suena por esa rama llega a golpes, uno por periodo (≈4 ms en una voz femenina), también las
fricativas y las respiraciones, que entonces se oyen ásperas, como estática. Este ajuste, solo de inferencia,
sustituye los pulsos por ruido de la misma potencia por encima de un armónico: el 2 en los tramos sordos
(«s», «f», «j», pausas) y el 33 en los sonoros (solo lo más agudo de las vocales). Los pesos no cambian.

Portado de stdevMac/tiny-voca (Apache 2.0), ``NOISE_ABOVE`` en src/tiny_voca/model.py, commit a4bf2bd
(docs/word-edge-artifact.md, sección 11). Se aplica sobre el tiny_voca oficial sin editar sus archivos:

    import ruido_fricativas
    ruido_fricativas.activar()            # (2, 33) por defecto; activar(None) vuelve a los pulsos solos
    # ...luego load_model / speak_text como siempre
"""
import math

import torch
from torch import nn

from tiny_voca import model as _model

NOISE_ABOVE = (2.0, 33.0)
_estado = {"noise_above": None}
_originales = {}


def _noise_share(self, harmonics, f0, here, following):
    """Reparto pulsos/ruido por sub-frame y bin: sube de 0 a 1 en la octava centrada en el armónico límite."""
    c = self.config
    hertz = torch.arange(c.synth_fft // 2 + 1, device=f0.device) * (c.sample_rate / c.synth_fft)
    octaves = torch.log2(hertz.clamp_min(1e-3) / (harmonics.float() * f0).clamp_min(1e-6)[..., None]).clamp(-0.5, 0.5)
    share = 0.5 * (1 + torch.sin(math.pi * octaves))
    band = (hertz >= 0.5 * f0[..., None]) & (hertz <= c.harmonic_limit * c.sample_rate)
    share, band = (0.5 * (self._take(value, here) + self._take(value, following)) for value in (share, band.float()))
    return share, band


def _forward(self, hidden, f0, lengths, noise=None, phase=None, noise_above=None, breath=None):
    """SourceFilter.forward del repo oficial + la mezcla de stdevMac cuando ``noise_above`` no es None."""
    F, cepstrum_response, length_mask, pulse_train = _model.F, _model.cepstrum_response, _model.length_mask, _model.pulse_train
    c = self.config
    frames = hidden.shape[1]
    samples = frames * c.hop
    with torch.autocast(device_type=hidden.device.type, enabled=False):
        hidden = hidden.float()
        audible = length_mask(lengths * c.hop, samples)
        last = (lengths - 1).clamp_min(0)[:, None]
        h = self.harmonic(hidden)
        split = c.cep_causal + c.cep_anti
        sub = self._subframes(frames, lengths, self.harmonic_steps)
        here, following = sub // self.harmonic_steps, torch.minimum((sub + 1) // self.harmonic_steps, last)
        causal = 0.5 * (self._take(h[..., :c.cep_causal], here) + self._take(h[..., :c.cep_causal], following)) * self.causal_lifter
        anti = 0.5 * (self._take(h[..., c.cep_causal:split], here) + self._take(h[..., c.cep_causal:split], following)) * self.anti_lifter
        causal = torch.cat((causal[..., :1] + h[..., split:].flatten(1).gather(1, sub)[..., None], causal[..., 1:]), dim=-1)
        harmonic_response = cepstrum_response(causal, anti, c.synth_fft)
        n = self.noise(hidden)
        sub = self._subframes(frames, lengths, self.noise_steps)
        shape = self._take(n[..., :c.noise_cep], sub // self.noise_steps) * self.noise_lifter
        shape = torch.cat((shape[..., :1] + n[..., c.noise_cep:].flatten(1).gather(1, sub)[..., None], shape[..., 1:]), dim=-1)
        noise_response = cepstrum_response(shape, None, c.noise_fft)
        f0 = f0.float().gather(1, torch.minimum(torch.arange(frames, device=f0.device)[None], last))
        pulses = pulse_train(f0, c.sample_rate, c.hop, c.harmonic_limit, phase) * audible
        if noise is None:
            noise = torch.randn(hidden.shape[0], samples, device=hidden.device)
        if noise_above is None:
            harmonic = self._shape(pulses, harmonic_response, c.synth_fft, c.synth_hop, self.synth_window)
        else:
            share, band = _noise_share(self, noise_above, f0, here, following)
            if breath is None:
                breath = torch.randn(hidden.shape[0], samples, device=hidden.device)
            harmonic = (self._shape(pulses, harmonic_response * torch.sqrt(1 - share), c.synth_fft, c.synth_hop, self.synth_window)
                        + self._shape(breath * audible * math.sqrt(0.5 / c.harmonic_limit), harmonic_response * torch.sqrt(share) * band,
                                      c.synth_fft, c.synth_hop, self.synth_window))
        wave = harmonic + self._shape(noise * audible, noise_response, c.noise_fft, c.noise_hop, self.noise_window)
        return wave * audible


@torch.inference_mode()
def _synthesize(self, tokens, stress, lengths, speakers, speed=1.0, pitch_shift=0.0, seed=0, pitch_range=1.0, pitch_jitter=0.0,
                voiced_vowels=True):
    """TinyVoca.synthesize del repo oficial; con el ajuste activo calcula el límite por frame (sordo/sonoro)."""
    if not math.isfinite(speed) or not 0.25 <= speed <= 4 or not math.isfinite(pitch_shift) or not 0 <= pitch_range <= 3:
        raise ValueError("speed must be between 0.25 and 4, pitch_shift finite and pitch_range between 0 and 3")
    hidden, mask = self.encode(tokens, stress, lengths, speakers)
    durations = self.predict_durations(hidden, mask, tokens, speed)
    frames, frame_mask, frame_lengths = self.expand(hidden, durations, mask, speakers)
    prosody_hidden, prosody = self.predict_prosody(frames, frame_mask)
    if voiced_vowels:
        ends = durations.cumsum(1)
        frame = torch.arange(frames.shape[1], device=tokens.device)[None].expand(len(lengths), -1).contiguous()
        owner = torch.searchsorted(ends, frame, right=True).clamp(max=tokens.shape[1] - 1)
        vowel = self.vowel_tokens[tokens].gather(1, owner) & frame_mask
        prosody = torch.cat((prosody[..., :1], torch.where(vowel, prosody[..., 1].clamp_min(2.0), prosody[..., 1])[..., None],
                             prosody[..., 2:]), dim=-1)
    prosody = self.widen_pitch(prosody, frame_mask, pitch_range)
    acoustic = self.decode(prosody_hidden, self.hard_prosody(prosody), frame_mask)
    # Mismo ruido que el oficial (mismo generador y semilla); el ruido nuevo sale después del mismo generador.
    streams = [torch.Generator().manual_seed(seed + row) for row in range(len(frame_lengths))]
    noise = nn.utils.rnn.pad_sequence([torch.randn(int(n) * self.config.hop, generator=stream)
                                       for stream, n in zip(streams, frame_lengths)], batch_first=True)
    cutoff = breath = None
    if _estado["noise_above"] is not None:
        unvoiced, voiced = _estado["noise_above"]
        cutoff = torch.where(prosody[..., 1] > 0, float(voiced), float(unvoiced))
        breath = nn.utils.rnn.pad_sequence([torch.randn(int(n) * self.config.hop, generator=stream)
                                            for stream, n in zip(streams, frame_lengths)], batch_first=True).to(hidden.device)
    pitch = prosody[..., 0] + pitch_shift
    if pitch_jitter > 0:
        pitch = pitch + self.pitch_wobble(pitch.shape, pitch_jitter, seed, pitch.device)
    wave = self.vocoder(acoustic, self.frequency(pitch), frame_lengths, noise=noise.to(hidden.device),
                        noise_above=cutoff, breath=breath)
    return wave, frame_lengths * self.config.hop, {"durations": durations, "prosody": prosody, "frame_mask": frame_mask}


def activar(noise_above=NOISE_ABOVE):
    """Activa el ajuste (pareja: armónico límite en tramos sordos, en tramos sonoros). None = pulsos solos."""
    if not _originales:
        _originales.update(forward=_model.SourceFilter.forward, synthesize=_model.TinyVoca.synthesize)
        _model.SourceFilter.forward = _forward
        _model.TinyVoca.synthesize = _synthesize
    _estado["noise_above"] = None if noise_above is None else (float(noise_above[0]), float(noise_above[1]))
'''
(REPO_SRC / "ruido_fricativas.py").write_text(RUIDO_FRICATIVAS_PY, encoding="utf-8")
shutil.copyfile(ALIGN / "codigo" / "frontend_es.py", REPO_SRC / "frontend_es.py")
(REPO_SRC / "tiny_voca" / "text.py").write_text(TEXT_ES, encoding="utf-8")
# model.py crea un Config() por defecto al importarse: su vocab_size por defecto (76, el inglés) debe ser el nuestro.
config_py = REPO_SRC / "tiny_voca" / "config.py"
source = config_py.read_text(encoding="utf-8")
if "vocab_size: int = 76" in source:
    config_py.write_text(source.replace("vocab_size: int = 76", "vocab_size: int = 56  # español (notebook 03)"),
                         encoding="utf-8")
elif "vocab_size: int = 56" not in source:
    raise RuntimeError("No encuentro vocab_size en tiny_voca/config.py: el repo oficial ha cambiado")
(RUN / "codigo").mkdir(exist_ok=True)
for source in (REPO_SRC / "frontend_es.py", REPO_SRC / "tiny_voca" / "text.py", REPO_SRC / "ruido_fricativas.py"):
    shutil.copyfile(source, RUN / "codigo" / ("text_es.py" if source.name == "text.py" else source.name))
sys.path.insert(0, str(REPO_SRC))
for name in [m for m in sys.modules if m in ("frontend_es", "ruido_fricativas") or m.startswith("tiny_voca")]:
    del sys.modules[name]
import frontend_es as fe
from tiny_voca.config import Config
from tiny_voca import text as tv_text

first = json.loads(next(line for line in (ANNOTATIONS / "validation.jsonl").open(encoding="utf-8") if line.strip()))
if first["frontend"] != tv_text.frontend_fingerprint():
    raise RuntimeError("El front end no coincide con el de las anotaciones del 02")
CONFIG = Config(speakers=(first["speaker"],), vocab_size=len(fe.SYMBOLS))
if first["acoustic"] != CONFIG.acoustic_fingerprint:
    raise RuntimeError("Los rasgos acústicos del 02 no coinciden con esta versión del repo")
CONFIG_PATH = RUN / "config_modelo.json"
if CONFIG_PATH.exists():
    if Config.from_dict(json.loads(CONFIG_PATH.read_text())) != CONFIG:   # reanudar: misma configuración
        raise RuntimeError(f"{CONFIG_PATH} es de otra configuración: usa otro RUN_NAME")
else:
    CONFIG_PATH.write_text(json.dumps(CONFIG.to_dict(), indent=2))
print(f"tiny-voca {COMMIT[:10]} · {len(fe.SYMBOLS)} símbolos · voz {CONFIG.speakers[0]} · "
      f"huella {tv_text.frontend_fingerprint()[:12]}")

# Anotaciones y rasgos a disco local (leer de Drive durante el entrenamiento sería muy lento).
# Se copian solo los .pt de los clips aceptados; el manifiesto se copia byte a byte (su huella cuenta).
(DATA_LOCAL / "anotaciones").mkdir(parents=True, exist_ok=True)
paths = set()
for split in ("train", "validation", "test"):
    shutil.copyfile(ANNOTATIONS / f"{split}.jsonl", DATA_LOCAL / "anotaciones" / f"{split}.jsonl")
    for line in (ANNOTATIONS / f"{split}.jsonl").open(encoding="utf-8"):
        if line.strip():
            paths.add(json.loads(line)["path"])
local_meta = {**metadata, "acoustic_root": str(DATA_LOCAL / "features")}
(DATA_LOCAL / "anotaciones" / "annotations.json").write_text(json.dumps(local_meta, indent=2))
(DATA_LOCAL / "features").mkdir(exist_ok=True)
(DATA_LOCAL / "lista.txt").write_text("\n".join(sorted(paths)) + "\n")
started = time.time()
print(f"copiando {len(paths)} clips de Drive a disco local…")
subprocess.run(["rsync", "-a", "--files-from", str(DATA_LOCAL / "lista.txt"), f"{metadata['acoustic_root']}/",
                f"{DATA_LOCAL / 'features'}/"], check=True)
missing = [p for p in paths if not (DATA_LOCAL / "features" / p).is_file()]
if missing:
    raise RuntimeError(f"faltan {len(missing)} archivos de rasgos, p. ej. {missing[:3]}")
print(f"listo en {round(time.time() - started)} s")

In [ ]:
#@title 3 · Entrenar (reanudable; para con ■ y vuelve a ejecutar para seguir)
import threading

STOP_FILE = RUN_LOCAL / "PARAR"


def sync(source, target):
    result = subprocess.run(["rsync", "-a", "--update", "--exclude", "*.tmp", "--exclude", "PARAR", f"{source}/", f"{target}/"],
                            capture_output=True, text=True)
    if result.returncode:
        print("  ⚠ no se pudo copiar a Drive (se reintenta luego):", result.stderr.strip()[-200:])
    return result.returncode == 0


# Retomar: lo de Drive a disco local (sin pisar lo local más nuevo).
RUN_LOCAL.mkdir(exist_ok=True)
STOP_FILE.unlink(missing_ok=True)
sync(RUN, RUN_LOCAL)
resume = RUN_LOCAL / "last.pt"
if not resume.exists():
    # Empezar de cero: el entrenador exige una carpeta de salida vacía (lo de Drive se queda como está).
    shutil.rmtree(RUN_LOCAL)
    RUN_LOCAL.mkdir()
done_steps = torch.load(resume, map_location="cpu", weights_only=True)["step"] if resume.exists() else 0
if done_steps >= STEPS:
    print(f"Ya está entrenado hasta el paso {done_steps} (STEPS = {STEPS}). Sube STEPS para seguir, o escucha en la celda 4.")
else:
    command = [sys.executable, "-u", "-m", "tiny_voca.train",
               "--train", str(DATA_LOCAL / "anotaciones" / "train.jsonl"),
               "--validation", str(DATA_LOCAL / "anotaciones" / "validation.jsonl"),
               "--output", str(RUN_LOCAL), "--config", str(CONFIG_PATH), "--steps", str(STEPS),
               "--save-every", str(SAVE_EVERY), "--keep-every", str(KEEP_EVERY), "--log-every", "50",
               "--max-seconds", str(30 * 24 * 3600), "--stop-file", str(STOP_FILE),
               *[str(x) for item in RECIPE.items() for x in item]]
    if resume.exists():
        command += ["--resume", str(resume)]
        print(f"Reanudando desde el paso {done_steps}.")
    else:
        print("Empezando desde cero.")
    environment = {**os.environ, "PYTHONPATH": str(REPO_SRC)}
    process = subprocess.Popen(command, env=environment, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                               bufsize=1, start_new_session=True)   # ■ llega al notebook, no al entrenamiento
    last_sync, last_print, first = time.time(), 0.0, None
    try:
        for line in process.stdout:
            try:
                record = json.loads(line)
            except json.JSONDecodeError:
                print(line.rstrip())
                continue
            if "inference_parameters" in record:
                print(f"parámetros del modelo: {record['inference_parameters']:,} · clips de train: {record['train_utterances']}")
                continue
            step = record["step"]
            first = first or (step, time.time())
            validation = "validation_predicted_mel" in record
            if validation or time.time() - last_print > 120:
                rate = (step - first[0]) / max(time.time() - first[1], 1e-9)
                left = f"~{(STEPS - step) / rate / 3600:.1f} h" if rate > 0 else "(calculando)"
                text = (f"paso {step:>6}/{STEPS} · pérdida {record['loss']:.3f} · mel {record.get('mel', float('nan')):.3f} · "
                        f"{record['steps_per_second']:.2f} pasos/s · quedan {left}")
                if validation:
                    text += (f"\n   validación: mel {record['validation_predicted_mel']:.3f} · duración "
                             f"{record['validation_duration']:.3f} · tono {record['validation_pitch_error_semitones']:.2f} st"
                             f" → guardado inference.pt")
                print(text, flush=True)
                last_print = time.time()
            if validation and time.time() - last_sync > SYNC_MINUTES * 60:
                threading.Thread(target=sync, args=(RUN_LOCAL, RUN), daemon=True).start()
                last_sync = time.time()
        process.wait()
    except KeyboardInterrupt:
        print("Parando: el entrenamiento guarda en el siguiente paso y sale…")
        STOP_FILE.touch()
        process.wait()
    finally:
        STOP_FILE.unlink(missing_ok=True)
        print("Copiando a Drive…")
        while not sync(RUN_LOCAL, RUN):
            time.sleep(30)
    if process.returncode not in (0, None):
        raise RuntimeError(f"El entrenamiento falló (código {process.returncode}); mira el error de arriba. "
                           "Si dice «CUDA out of memory», usa otro RUN_NAME con '--batch-frames' más bajo (p. ej. 6000).")
    step = torch.load(resume, map_location="cpu", weights_only=True)["step"] if resume.exists() else 0
    print(f"Guardado en Drive hasta el paso {step}: {RUN}")

In [ ]:
#@title 4 · Escuchar el modelo (último inference.pt guardado)
import random
import soundfile as sf
from IPython.display import Audio, Markdown, display
from tiny_voca.infer import load_model, speak_text
import ruido_fricativas
ruido_fricativas.activar() if RUIDO_FRICATIVAS else ruido_fricativas.activar(None)

checkpoint = RUN_LOCAL / "inference.pt" if (RUN_LOCAL / "inference.pt").exists() else RUN / "inference.pt"
if not checkpoint.exists():
    raise FileNotFoundError("Todavía no hay inference.pt: entrena al menos SAVE_EVERY pasos en la celda 3")
model = load_model(checkpoint, "cuda")
step = torch.load(checkpoint, map_location="cpu", weights_only=True)["step"]
samples = RUN / "muestras" / f"paso-{step:06d}"
samples.mkdir(parents=True, exist_ok=True)
test_rows = [json.loads(line) for line in (DATA_LOCAL / "anotaciones" / "test.jsonl").open(encoding="utf-8") if line.strip()]
chosen = random.Random(step).sample(test_rows, min(3, len(test_rows)))
display(Markdown(f"### Paso {step}"))
for number, sentence in enumerate(LISTEN_SENTENCES + [row["text"] for row in chosen], 1):
    with torch.no_grad():
        wave, _ = speak_text(model, sentence, CONFIG.speakers[0])
    sf.write(str(samples / f"{number:02d}.wav"), wave.numpy(), CONFIG.sample_rate)
    display(Markdown(f"**{sentence}**" + (" *(frase de test: no la vio al entrenar)*" if number > len(LISTEN_SENTENCES) else "")))
    display(Audio(wave.numpy(), rate=CONFIG.sample_rate))
print("guardado en", samples)

In [ ]:
#@title 5 · Apagar el entorno al terminar (deja de gastar unidades)
# Con «Ejecutar todo» corre después de entrenar y escuchar. Si paraste la celda 3 con ■ no se ejecuta sola.
from google.colab import drive, runtime
drive.flush_and_unmount()   # asegura que todo lo escrito llegó a Drive
runtime.unassign()          # libera la GPU: Colab se desconecta y deja de cobrar

## Qué queda en Drive

`03_entrenamiento/<RUN_NAME>/`:

* `inference.pt`: **el modelo** (≈ 18 MB). Es lo que necesitas para usarlo.
* `last.pt` y `step-XXXXXXXX.pt`: estado completo para reanudar (no hacen falta para usarlo).
* `metrics.jsonl`, `config.json`, `recipe.json`, `config_modelo.json`: registro del entrenamiento.
* `codigo/frontend_es.py` y `codigo/text_es.py`: el front end español. Para usar el modelo fuera de este
  notebook, clona el repo oficial, copia `frontend_es.py` junto a la carpeta `tiny_voca/` (en `src/`) y
  `text_es.py` encima de `tiny_voca/text.py`. Después funciona igual que en inglés:
  `python -m tiny_voca.infer --checkpoint inference.pt --voice F2 --text "Hola." --output hola.wav`
  (escribe los números con letra).
* `codigo/ruido_fricativas.py`: ajuste de inferencia que quita la estática de las «s», «f» y respiraciones
  (portado de stdevMac/tiny-voca, Apache 2.0). Ponlo también en `src/` y, en Python, antes de generar:
  `import ruido_fricativas; ruido_fricativas.activar()`. El comando `python -m tiny_voca.infer` no lo usa.
* `muestras/paso-XXXXXX/`: lo que generó la celda 4 en cada momento.